In [7]:
import pandas as pd
import os
print(os.getcwd())

df = pd.read_parquet('demo_bronze.parquet')
df

/Users/kevinkurek/Desktop/github/opra_to_alpha/research


,symbol,msg_count
0,AAPL240927C00190000,42


In [10]:
import json, struct, datetime

PATH = "../rust-ingest/pcap_samples/example_packets.json"

# --- helpers ---------------------------------------------------------------
def read_payload(pkt):
    layers = (pkt.get("_source") or pkt).get("layers", {})
    s = layers.get("udp", {}).get("udp.payload") or layers.get("data", {}).get("data.data")
    if isinstance(s, list): s = s[0]
    return bytes.fromhex(s.replace(":", "")) if s else b""

def deno(code: str) -> int:
    # A=1 digit right of decimal, B=2, ... (extend as needed from spec table)
    return {"A":1, "B":2}.get(code, 0)

def as_price(raw: int, code: str) -> float:
    d = deno(code)
    return raw / (10 ** d) if d else float(raw)

# --- parsers ---------------------------------------------------------------
def parse_block(buf: bytes, off=0):
    # Block Header (21 bytes) — Version(1), BlockSize(2), DFI(1), Retrans(1), Session(1),
    # BlockSeq(4), MsgsInBlock(1), BlockTs(8), Checksum(2)  [network byte order]
    ver = buf[off]; off += 1
    blk_size = struct.unpack_from(">H", buf, off)[0]; off += 2
    dfi = chr(buf[off]); off += 1
    retrans = chr(buf[off]); off += 1
    session = buf[off]; off += 1
    blk_seq = struct.unpack_from(">I", buf, off)[0]; off += 4
    msgs = buf[off]; off += 1
    ts_ns_hi, ts_ns_lo = struct.unpack_from(">II", buf, off); off += 8
    chk = struct.unpack_from(">H", buf, off)[0]; off += 2
    ts_ns = (ts_ns_hi << 32) | ts_ns_lo
    # OPRA timestamp is nanoseconds from Unix epoch (per feed conventions)
    ts = datetime.datetime.utcfromtimestamp(ts_ns / 1e9)
    header = dict(version=ver, block_size=blk_size, data_feed_indicator=dfi,
                  retransmission_indicator=retrans, session_indicator=session,
                  block_sequence=blk_seq, messages_in_block=msgs,
                  block_timestamp_utc=str(ts), checksum=chk)
    return header, off

def parse_msg_header(buf: bytes, off: int):
    # 12-byte Message Header: ParticipantID(1), Category(1), Type(1), Indicator(1),
    # TransactionID(4), ParticipantRefNum(4)
    pid = chr(buf[off]); cat = chr(buf[off+1]); typ = chr(buf[off+2]); ind = chr(buf[off+3])
    txn, prn = struct.unpack_from(">II", buf, off+4)
    return {"participant": pid, "category": cat, "type": typ, "indicator": ind,
            "transaction_id": txn, "prn": prn}, off + 12

def parse_quote_k(buf: bytes, off: int):
    # Long Quote (k): Security(5 ASCII), Reserved(1), ExpBlock(3),
    # StrikeDen(1 ASCII), Strike(4 i32), PremiumDen(1 ASCII),
    # Bid(4 i32), BidSz(4 u32), Ask(4 i32), AskSz(4 u32)  => 43 bytes
    sym = buf[off:off+5].decode("ascii").rstrip(); off += 5
    off += 1  # reserved
    exp = buf[off:off+3]; off += 3
    strike_den = chr(buf[off]); off += 1
    strike = struct.unpack_from(">i", buf, off)[0]; off += 4
    prem_den = chr(buf[off]); off += 1
    bid  = struct.unpack_from(">i", buf, off)[0]; off += 4
    bids = struct.unpack_from(">I", buf, off)[0]; off += 4
    ask  = struct.unpack_from(">i", buf, off)[0]; off += 4
    asks = struct.unpack_from(">I", buf, off)[0]; off += 4
    return {
        "symbol": sym, "expiration_block": exp.hex(),
        "strike": as_price(strike, strike_den),
        "bid": as_price(bid,  "B"),   # Premium denom implied B (2 dp) for quotes
        "bid_size": bids,
        "ask": as_price(ask,  "B"),
        "ask_size": asks,
        "msg_len": 43,
        "strike_den": strike_den, "premium_den": "B"
    }, off

def parse_quote_q(buf: bytes, off: int):
    # Short Quote (q): Header + Security(4 ASCII), ExpBlock(3), Strike(2 u16),
    # Bid(2 u16), BidSz(2 u16), Ask(2 u16), AskSz(2 u16) => 29 bytes
    sym = buf[off:off+4].decode("ascii").rstrip(); off += 4
    exp = buf[off:off+3]; off += 3
    strike, bid, bids, ask, asks = struct.unpack_from(">HHHHH", buf, off); off += 10
    # Short quote StrikeDen implied 'A' (1 dp); PremiumDen implied 'B' (2 dp)
    return {
        "symbol": sym, "expiration_block": exp.hex(),
        "strike": as_price(strike, "A"),
        "bid": as_price(bid,  "B"),
        "bid_size": bids,
        "ask": as_price(ask,  "B"),
        "ask_size": asks,
        "msg_len": 29,
        "strike_den": "A", "premium_den": "B"
    }, off

def decode_udp_payload(buf: bytes):
    out = {}
    bh, off = parse_block(buf, 0)                 # ---- Block Header (21 bytes)
    out["block_header"] = bh
    msgs = []
    for _ in range(bh["messages_in_block"]):
        mh, off = parse_msg_header(buf, off)      # ---- 12-byte Message Header
        rec = {"header": mh}
        cat = mh["category"]
        if cat == "k":
            body, off = parse_quote_k(buf, off)
            rec["body"] = body
        elif cat == "q":
            body, off = parse_quote_q(buf, off)
            rec["body"] = body
        else:
            rec["body"] = {"raw_hex": buf[off:off+32].hex(), "note": "parser: category not implemented"}
            # Try to infer length from spec if needed; here we just peek 32 bytes.
        msgs.append(rec)
    out["messages"] = msgs
    return out

# --- run on first packet in your file --------------------------------------
with open(PATH, "r") as f:
    data = json.load(f)
if isinstance(data, dict): data = [data]
payload = read_payload(data[0])
decoded = decode_udp_payload(payload)

from pprint import pprint
pprint(decoded)

{'block_header': {'block_sequence': 355512323,
                  'block_size': 108,
                  'block_timestamp_utc': '2200-05-20 07:10:45.166555',
                  'checksum': 7792,
                  'data_feed_indicator': 'O',
                  'messages_in_block': 3,
                  'retransmission_indicator': ' ',
                  'session_indicator': 0,
                  'version': 6},
 'messages': [{'body': {'ask': 0.5,
                        'ask_size': 418,
                        'bid': 0.48,
                        'bid_size': 310,
                        'expiration_block': '541e17',
                        'msg_len': 29,
                        'premium_den': 'B',
                        'strike': 423.0,
                        'strike_den': 'A',
                        'symbol': 'SPY'},
               'header': {'category': 'q',
                          'indicator': 'A',
                          'participant': 'C',
                          'prn': 999501000,


In [24]:
import json, struct, datetime, pandas as pd

PATH = "../rust-ingest/pcap_samples/example_packets5.json"

# ---------- Helpers ----------
def read_payload(pkt):
    layers = (pkt.get("_source") or pkt).get("layers", {})
    s = layers.get("udp", {}).get("udp.payload") or layers.get("data", {}).get("data.data")
    if isinstance(s, list): s = s[0]
    return bytes.fromhex(s.replace(":", "")) if s else b""

def deno(code: str) -> int: return {"A":1, "B":2}.get(code, 0)
def as_price(raw: int, code: str) -> float:
    d = deno(code); return raw / (10 ** d) if d else float(raw)

# ---------- Decode helpers ----------
_MONTH_CALL = {c:i for i,c in enumerate("ABCDEFGHIJKL", start=1)}  # Jan–Dec calls
_MONTH_PUT  = {c:i for i,c in enumerate("MNOPQRSTUVWX", start=1)}  # Jan–Dec puts

def decode_exp_block(exp_bytes: bytes):
    """Decode OPRA 3-byte expiration block → (YYMMDD, 'C'/'P')"""
    mchr = chr(exp_bytes[0]); day = exp_bytes[1]; yy = exp_bytes[2]
    if mchr in _MONTH_CALL: month = _MONTH_CALL[mchr]; cp = "C"
    elif mchr in _MONTH_PUT: month = _MONTH_PUT[mchr]; cp = "P"
    else: return "YYMMDD", "?"
    year = 2000 + yy
    return f"{year%100:02d}{month:02d}{day:02d}", cp

def to_osi_symbol(root: str, yymmdd: str, cp: str, strike: float) -> str:
    strike_int = int(round(strike * 1000))
    return f"{root} {yymmdd}{cp}{strike_int:08d}"

# ---------- Parsers ----------
def parse_block(buf: bytes, off=0):
    ver = buf[off]; off += 1
    blk_size = struct.unpack_from(">H", buf, off)[0]; off += 2
    dfi = chr(buf[off]); off += 1
    retrans = chr(buf[off]); off += 1
    session = buf[off]; off += 1
    blk_seq = struct.unpack_from(">I", buf, off)[0]; off += 4
    msgs = buf[off]; off += 1
    sec, nsec = struct.unpack_from(">II", buf, off); off += 8       # fixed
    chk = struct.unpack_from(">H", buf, off)[0]; off += 2
    ts = datetime.datetime.utcfromtimestamp(sec + nsec / 1e9)
    header = dict(
        version=ver, block_size=blk_size, data_feed_indicator=dfi,
        retransmission_indicator=retrans, session_indicator=session,
        block_sequence=blk_seq, messages_in_block=msgs,
        block_timestamp_utc=ts.replace(tzinfo=datetime.timezone.utc).isoformat(),
        checksum=chk
    )
    return header, off

def parse_msg_header(buf: bytes, off: int):
    pid = chr(buf[off]); cat = chr(buf[off+1]); typ = chr(buf[off+2]); ind = chr(buf[off+3])
    txn, prn = struct.unpack_from(">II", buf, off+4)
    return dict(participant=pid, category=cat, type=typ, indicator=ind,
                transaction_id=txn, prn=prn), off + 12

def parse_quote_q(buf: bytes, off: int):
    sym = buf[off:off+4].decode("ascii").rstrip(); off += 4
    exp = buf[off:off+3]; off += 3
    strike, bid, bids, ask, asks = struct.unpack_from(">HHHHH", buf, off); off += 10
    yymmdd, cp = decode_exp_block(exp)
    strike_val = as_price(strike, "A")
    return {
        "symbol_root": sym,
        "yymmdd": yymmdd,
        "cp": cp,
        "strike": strike_val,
        "bid": as_price(bid, "B"),
        "ask": as_price(ask, "B"),
        "bid_size": bids,
        "ask_size": asks,
        "osi_symbol": to_osi_symbol(sym, yymmdd, cp, strike_val)
    }, off

def decode_udp_payload(buf: bytes):
    bh, off = parse_block(buf, 0)
    records = []
    for _ in range(bh["messages_in_block"]):
        mh, off = parse_msg_header(buf, off)
        if mh["category"] != "q":  # only short quotes here
            continue
        body, off = parse_quote_q(buf, off)
        records.append({**bh, **mh, **body})
    return records

# ---------- Run all packets ----------
data = json.load(open(PATH))
if isinstance(data, dict): data = [data]

rows = []
for pkt in data[:100]:  # adjust slice as needed
    buf = read_payload(pkt)
    rows.extend(decode_udp_payload(buf))

df = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
df[df['osi_symbol'] == 'SPY 230830P00423000']

,version,block_size,data_feed_indicator,retransmission_indicator,session_indicator,block_sequence,messages_in_block,block_timestamp_utc,checksum,participant,category,type,indicator,transaction_id,prn,symbol_root,yymmdd,cp,strike,bid,ask,bid_size,ask_size,osi_symbol
0,6,108,O,,0,355512323,3,2023-08-22T14:29:59.999801+00:00,7792,C,q,A,A,1091292227,999501000,SPY,230830,P,423.0,0.48,0.5,310,418,SPY 230830P00423000


In [26]:
import os
import databento as db
from dotenv import load_dotenv

load_dotenv('./.env')  # take environment variables

client = db.Historical(os.getenv("DATABENTO_API_KEY"))  # uses DATABENTO_API_KEY env var
df = client.timeseries.get_range(
    dataset="OPRA.PILLAR",
    schema="cmbp-1",                
    # symbols=["SPY.OPT"], 
    # stype_in="parent",                   # request the chain; filter in code
    stype_in="raw_symbol",
    symbols=["SPY   230830P00423000"],
    start="2023-08-22T14:29:59.990000Z",
    end="2023-08-22T14:30:00.500000Z",
    limit=100
).to_df()
df

,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_pb_00,ask_pb_00,symbol
ts_recv,,,,,,,,,,,,,,,,,
2023-08-22 14:29:59.991083538+00:00,2023-08-22 14:29:59.990872832+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1497,271,0,0,SPY 230830P00423000
2023-08-22 14:30:00.048469173+00:00,2023-08-22 14:30:00.048260352+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1670,271,0,0,SPY 230830P00423000
2023-08-22 14:30:00.075664939+00:00,2023-08-22 14:30:00.075455488+00:00,177,30,654312856,A,N,0.49,321,194,0,0.48,0.49,1670,321,0,0,SPY 230830P00423000
2023-08-22 14:30:00.106431944+00:00,2023-08-22 14:30:00.106223616+00:00,177,30,654312856,A,N,0.49,321,194,0,0.48,0.49,1423,321,0,0,SPY 230830P00423000
2023-08-22 14:30:00.449679234+00:00,2023-08-22 14:30:00.449470208+00:00,177,30,654312856,A,N,0.49,271,194,0,0.48,0.49,1423,271,0,0,SPY 230830P00423000
2023-08-22 14:30:00.452095094+00:00,2023-08-22 14:30:00.451885824+00:00,177,30,654312856,A,N,0.49,92,194,0,0.48,0.49,1423,92,0,0,SPY 230830P00423000
2023-08-22 14:30:00.488356506+00:00,2023-08-22 14:30:00.488147200+00:00,177,30,654312856,A,N,0.49,124,194,0,0.48,0.49,1423,124,0,0,SPY 230830P00423000
2023-08-22 14:30:00.488361322+00:00,2023-08-22 14:30:00.488152064+00:00,177,30,654312856,A,N,0.49,92,194,0,0.48,0.49,1423,92,0,0,SPY 230830P00423000
